*This code is a companion to the book **Mastering PyTorch and Lightning: A Step-by-Step Practical Guide with QA** by Aghiles Kebaili*
> This notebook contains the raw code for Chapter 10: Streamlining Data with LightningDataModule. It focuses on reusable data preparation, deterministic splits, and cleaner stage-specific loading.

A well-structured DataModule keeps the dataset lifecycle explicit: prepare once on disk, build process-local state in setup, and expose the correct loader for each training stage.

## Implementing a CIFAR-10 DataModule
### From step 1 to step 4

In [ ]:
import os
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import pytorch_lightning as pl

# The DataModule separates configuration from process-local dataset state.
class VisionDataModule(pl.LightningDataModule):
    # Step 1: Store Configuration in __init__()
    def __init__(
        self,
        data_dir: str = "./data",
        batch_size: int = 256,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = min(4, os.cpu_count() or 1)
        self.pin_memory = torch.cuda.is_available()

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(
                (0.5, 0.5, 0.5),
                (0.5, 0.5, 0.5),
            ),
        ])

    # Step 2: Reserve prepare_data() for External Side Effects
    def prepare_data(self):
        datasets.CIFAR10(
            self.data_dir, train=True, download=True
        )
        datasets.CIFAR10(
            self.data_dir, train=False, download=True
        )

    # Step 3: Build Stage-Specific State in setup()
    def setup(self, stage: str | None = None):
        if stage in ("fit", "validate", None):
            if not hasattr(self, "cifar_train"):
                full_dataset = datasets.CIFAR10(
                    self.data_dir,
                    train=True,
                    transform=self.transform,
                )
                generator = torch.Generator().manual_seed(42)
                self.cifar_train, self.cifar_val = random_split(
                    full_dataset,
                    [45000, 5000],
                    generator=generator,
                )

        if stage in ("test", "predict", None):
            if not hasattr(self, "cifar_test"):
                self.cifar_test = datasets.CIFAR10(
                    self.data_dir,
                    train=False,
                    transform=self.transform,
                )

    # Step 4: Expose One Loader per Managed Stage
    def _make_loader(self, dataset, shuffle):
        return DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=shuffle,
            num_workers=self.num_workers,
            pin_memory=self.pin_memory,
            persistent_workers=self.num_workers > 0,
        )

    def train_dataloader(self):
        return self._make_loader(self.cifar_train, shuffle=True)

    def val_dataloader(self):
        return self._make_loader(self.cifar_val, shuffle=False)

    def test_dataloader(self):
        return self._make_loader(self.cifar_test, shuffle=False)

    def predict_dataloader(self):
        return self._make_loader(self.cifar_test, shuffle=False)

### Execute Preparation and Fit Setup Manually

In [4]:
datamodule = VisionDataModule(
    data_dir="./data",
    batch_size=64,
)

datamodule.prepare_data()

assert not hasattr(datamodule, "cifar_train")
assert not hasattr(datamodule, "cifar_val")

datamodule.setup(stage="fit")

assert len(datamodule.cifar_train) == 45000
assert len(datamodule.cifar_val) == 5000

train_loader = datamodule.train_dataloader()
images, labels = next(iter(train_loader))

assert images.shape == (64, 3, 32, 32)
assert labels.shape == (64,)

100%|██████████| 170M/170M [25:02<00:00, 114kB/s]  


### Step 7: Verify Split Reproducibility and Stage Isolation

In [ ]:
replica = VisionDataModule(
    data_dir="./data",
    batch_size=64,
)
replica.setup(stage="fit")

assert datamodule.cifar_train.indices == (
    replica.cifar_train.indices
)
assert datamodule.cifar_val.indices == (
    replica.cifar_val.indices
)

datamodule.setup(stage="test")
test_loader = datamodule.test_dataloader()
test_images, test_labels = next(iter(test_loader))

assert len(datamodule.cifar_test) == 10000
assert test_images.shape == (64, 3, 32, 32)
assert test_labels.shape == (64,)

## Reproducibility & Team Collaboration

### Step 1: Pair the Pipeline with a Compatible Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl

# The model and DataModule remain separate: the module owns the math, while the DataModule owns the dataset lifecycle.
class LitCIFARClassifier(pl.LightningModule):
    def __init__(self, lr: float = 1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        return self.backbone(x)

    def _shared_step(self, batch, prefix):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        self.log(
            f"{prefix}_loss",
            loss,
            on_epoch=True,
            prog_bar=True,
            sync_dist=prefix != "train",
            batch_size=x.size(0),
        )
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        self._shared_step(batch, "test")

    def predict_step(self, batch, batch_idx):
        x, _ = batch
        return self(x).argmax(dim=1)

    def configure_optimizers(self):
        return torch.optim.AdamW(
            self.parameters(), lr=self.hparams.lr
        )

model = LitCIFARClassifier(lr=1e-3)
with torch.inference_mode():
    smoke_logits = model(images[:4])

assert smoke_logits.shape == (4, 10)

### Step 2: Compose the Final Training Script

In [8]:
import torch
import pytorch_lightning as pl

model = LitCIFARClassifier(lr=1e-3)

use_cuda = torch.cuda.is_available()
# Enable only after benchmarking this model and workload.
use_compile = False
if use_compile:
    model.backbone = torch.compile(model.backbone)

datamodule = VisionDataModule(
    data_dir="./data",
    batch_size=512,
)

precision = "16-mixed" if use_cuda else "32-true"
trainer = pl.Trainer(
    max_epochs=20,
    accelerator="auto",
    devices="auto",
    precision=precision,
)

trainer.fit(model, datamodule=datamodule)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3060') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type       | Params | Mode 
------------------------------------------------
0 | backbone | Sequential | 1.2 K  | train
------------------------------------------------
1.2 K     Trainable params
0         Non-trainable params
1.2 K     Total params
0.005     Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.
